# Compare vector, hybrid, and semantic-ranker queries on the same Azure AI Search Basic index and capture the relevance scores

Your search team is debating whether pure vector, hybrid (vector plus BM25), or the semantic ranker is the right default for production. The product owner wants a data-backed answer before signing off on Standard tier ($245.28/month vs $73.73/month for Basic). The founder is impatient. You have one session to stand up an Azure AI Search Basic index with 200 representative docs, run five representative queries through all three retrieval modes, capture the relevance scores, and produce a comparison table the team can argue over in Friday's planning meeting. The deliverable is the captured measurements, not a foregone conclusion.

What you will build:

- A Foundry hub plus project in `eastus`, plus a `text-embedding-3-small` deployment for embeddings.
- An Azure AI Search Basic service. **This is the cost-critical resource in this lab.** Basic tier bills $73.73/month prorated hourly, about $0.10/hour, regardless of traffic.
- One search index with a 1536-dim vector field on HNSW plus a semantic configuration over `title` and `content`.
- A 200-chunk seeded corpus embedded with `text-embedding-3-small` and uploaded to the index.
- Three query modes against the same index: vector-only, hybrid (vector plus BM25), and hybrid plus the semantic ranker.
- A 15-row comparison table (5 queries x 3 modes) reporting top-1 score, top-3 mean score, and top result's document ID for each row.

**Time.** Plan for about 60 minutes hands-on. Search service first-time create can take 3 to 8 minutes; budget up to 120 minutes total. The cleanup cell at the bottom tears every Azure resource down.

**Cost.** This is a Search lab, so cleanup matters. Azure AI Search Basic costs ten cents an hour whether you are using it or not, and an overnight slip eats $2.45 of credit. A clean two-hour session lands around $0.20 to $0.30. The `text-embedding-3-small` deployment at $0.02 per 1M tokens adds fractions of a cent. The whole point of the safety manifest and the atexit handler is to make the search-service teardown unforgettable. Run cleanup before you close the tab, every time.

This lab maps to AI-103 Domain 5: Implement information extraction solutions (13% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs this lab needs. Versions are
# pinned per LAB-CREATION-BLUEPRINT.md build rules.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents.
# - azure-search-documents is the data-plane SDK for index and document operations.
# - azure-mgmt-search is the management-plane SDK for Search service create and delete.

!pip install --quiet labrun-checks==0.6.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 azure-mgmt-search>=9.1.0 azure-search-documents>=11.5.0 openai>=2.0.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.
# The search service name appends a 6-char random suffix because Search service
# names are globally unique.

import atexit
import getpass
import json
import random
import statistics
import string
import time

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.core.credentials import AzureKeyCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.search.models import SearchService, Sku
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
)
from azure.search.documents.models import VectorizedQuery
from openai import AzureOpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "vector-hybrid-semantic-search"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
EMBED_DEPLOYMENT_NAME = f"labrun-{LAB_ID}-embed3s"
INDEX_NAME = "labrun-vhs-index"

EMBED_MODEL_NAME = "text-embedding-3-small"
EMBED_MODEL_VERSION = "1"
EMBED_MODEL_CAPACITY_TPM = 30  # sku.capacity unit is thousands of TPM
EMBED_DIMENSIONS = 1536

# Search service names are globally unique; append a random suffix so two
# students in the same subscription do not collide on the same name.
_rand_suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))
SEARCH_SERVICE_NAME = f"labrun-vhs-search-{_rand_suffix}"

# Resolved during setup and Task 1.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
SEARCH_ENDPOINT = None
AZURE_CREDENTIAL = None

# State populated by tasks for later checkpoints.
CORPUS_CHUNKS = []
MODE_A_RESULTS = {}   # keyed by query_id, value is list of result dicts
MODE_B_RESULTS = {}
MODE_C_RESULTS = {}
COMPARISON_TABLE = []  # list of 15 dicts (5 queries x 3 modes)

# Five seeded test queries shaped like product-manual questions.
TEST_QUERIES = [
    {"id": "Q1", "text": "how do I rotate the API key"},
    {"id": "Q2", "text": "SKU-A123-X troubleshooting"},
    {"id": "Q3", "text": "what is the default retention period"},
    {"id": "Q4", "text": "configuring multi-region failover"},
    {"id": "Q5", "text": "billing alerts and budgets"},
]

# Pricing for wallet checks. Pricing claims must match
# https://azure.microsoft.com/en-us/pricing/details/cognitive-services/openai-service/
# and https://azure.microsoft.com/en-us/pricing/details/search/
EMBED_PRICE_PER_1M_TOKENS_USD = 0.02
SEARCH_BASIC_HOURLY_USD = 0.10


In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15. Search Basic is hourly-billed; the
# orphan scan in the next cell MUST run before any resource is created so a
# leftover service from a prior session does not silently accrue cost.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required. Find yours with `az account show --query id -o tsv`.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")
print(f"Search service name (rand suffix): {SEARCH_SERVICE_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# Critical resource: ai_search_service (Basic tier bills $73.73/month
# regardless of traffic, prorated hourly). Per RESOURCE-SAFETY-SPEC Rule 4,
# the orphan scan BLOCKS execution if any prior tagged resources exist.
#
# Reverse-creation order: search index first (dropped automatically with the
# service but the manifest is explicit), then the search service (critical),
# then the embedding deployment, the project, hub, and resource group. The
# resource group delete is the cross-cutting safety net.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="search_index",
        id=INDEX_NAME,
        region=REGION,
        cli_delete_command=(
            f"az search index delete --service-name {SEARCH_SERVICE_NAME} "
            f"--name {INDEX_NAME} --yes"
        ),
    ),
    CleanupResource(
        type="ai_search_service",
        id=SEARCH_SERVICE_NAME,
        region=REGION,
        cli_delete_command=(
            f"az search service delete --resource-group {RESOURCE_GROUP} "
            f"--name {SEARCH_SERVICE_NAME} --yes"
        ),
    ),
    CleanupResource(
        type="aoai_deployment",
        id=EMBED_DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {EMBED_DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run. Search Basic bills $0.10/hour")
    print("even when idle, so leaving an orphan service running costs $2.45/day.")
    print("Run the cleanup cell at the bottom of this notebook, or manually:")
    print(f"  az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Stand up the Foundry stack, deploy the embedding model, create the Search Basic service, and load a 200-doc index

This is the heavy task. It bundles every infrastructure piece the comparison needs into one block so the three query-mode tasks that follow can stay focused on the comparison itself. The Foundry pattern is the same shape as Lab 01; the Search service pattern is the same shape as Lab 05.

Build it in this order:

1. Create the resource group, hub, and project with the lab tag. Hub provisioning is async (4 to 6 minutes); project provisioning is async (2 to 3 minutes). The cell waits for both.
2. Discover the hub's attached Azure OpenAI account and deploy `text-embedding-3-small` against it at Standard SKU, 30k TPM.
3. Call `SearchManagementClient.services.begin_create_or_update` to provision the Search service at Basic tier in `eastus`, with the lab tag on the service itself. Wait for `provisioningState=Succeeded`. The validator REJECTS Free tier (no real vector profile) and Standard or higher (cost overkill for this 200-doc workload).
4. Build the index with a 1536-dim vector field on HNSW (`hnsw-config`, `vector-profile`) and a semantic configuration named `default` over `title` and `content`.
5. Seed the 200-chunk corpus inline. The chunks intentionally cover the topics in `TEST_QUERIES` (API key rotation, the literal `SKU-A123-X`, retention period, multi-region failover, billing alerts) plus 195 filler chunks so the three query modes have something to actually choose between.
6. Embed all 200 chunks in one batched embeddings call and upload in batches of 50 via `SearchClient.upload_documents`.

**Why one batch call for embeddings.** OpenAI embeddings accept an `input` array. One round trip for 200 chunks is cheaper than 200 round trips in API-call accounting, and Azure's quota system counts requests separately from tokens.

**Why this is the cost-critical resource.** Once `begin_create_or_update` completes on the Search service, you are billing $0.10/hour. If you walk away and the kernel times out, the service keeps billing until you (or the resource-group atexit fallback, or you on the CLI tomorrow morning) delete it. Setup registers the service in `CLEANUP_MANIFEST` before the create even returns so the atexit handler can find it.


In [ ]:
# NBVAL_SKIP
# Task 1: Provision RG, hub, project, embedding deployment, Search Basic
# service, vector + semantic index, then seed and embed 200 chunks.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
search_mgmt_client = SearchManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

# Resource group, hub, project. Same pattern as Lab 01.
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()

# Discover the attached Azure OpenAI account.
for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
if not AOAI_ACCOUNT_NAME:
    print("Could not find an attached Azure OpenAI account on the hub.")
    raise SystemExit(1)

aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")
print(f"Endpoint: {AOAI_ACCOUNT_ENDPOINT}")

# Deploy text-embedding-3-small.
embed_payload = {
    "sku": {"name": "Standard", "capacity": EMBED_MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": EMBED_MODEL_NAME, "version": EMBED_MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the embedding deployment go through Succeeded purgatory...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, EMBED_DEPLOYMENT_NAME, embed_payload,
).result()
print(f"text-embedding-3-small deployment ready: {EMBED_DEPLOYMENT_NAME}")

# Search Basic service. This is the cost-critical create.
search_payload = SearchService(
    location=REGION,
    sku=Sku(name="basic"),
    tags=lab_tags,
    partition_count=1,
    replica_count=1,
    public_network_access="enabled",
)
print("Asking ARM to allocate an Azure AI Search Basic service, this takes 3 to 8 minutes...")
print(f"Once this returns, you are billing ${SEARCH_BASIC_HOURLY_USD:.2f}/hour until cleanup runs.")
search_service = search_mgmt_client.services.begin_create_or_update(
    RESOURCE_GROUP, SEARCH_SERVICE_NAME, search_payload,
).result()
SEARCH_ENDPOINT = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
print(f"Search service provisioned: {SEARCH_SERVICE_NAME}")
print(f"Endpoint: {SEARCH_ENDPOINT}")
print(f"SKU: {search_service.sku.name}")

# Build the index with vector + semantic configuration.
search_index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT, credential=AZURE_CREDENTIAL,
)
hnsw_config = HnswAlgorithmConfiguration(name="hnsw-config")
vector_profile = VectorSearchProfile(
    name="vector-profile",
    algorithm_configuration_name="hnsw-config",
)
vector_search = VectorSearch(algorithms=[hnsw_config], profiles=[vector_profile])

semantic_config = SemanticConfiguration(
    name="default",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIMENSIONS,
        vector_search_profile_name="vector-profile",
    ),
    SimpleField(name="source_page", type=SearchFieldDataType.Int32, filterable=True),
]
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
created_index = search_index_client.create_or_update_index(index)
print(f"Index ready: {created_index.name}")

# Seed the 200-chunk corpus inline. Five chunks intentionally cover the topics
# in TEST_QUERIES; 195 filler chunks are generic product-manual content.
CORPUS_CHUNKS = []
seeded_topics = {
    7: ("Rotating the API key",
        "To rotate the API key, sign in to the admin console, open Settings, "
        "select Security, and click Rotate Key. The current key remains valid for "
        "24 hours so deployments have time to redeploy with the new value. "
        "Rotation is a manual operation; there is no scheduled auto-rotation."),
    23: ("Troubleshooting SKU-A123-X",
        "SKU-A123-X is the entry-level diagnostic kit. If the device shows a "
        "blinking amber LED, factory-reset by holding the side button for 10 "
        "seconds. SKU-A123-X firmware updates ship monthly through the device "
        "console; check Settings > Firmware on the SKU-A123-X dashboard."),
    44: ("Default retention period",
        "Audit logs are retained for 90 days by default. Customers on the "
        "Enterprise plan can extend retention up to 365 days through the "
        "Settings > Compliance > Retention panel. The retention period applies "
        "to both event logs and access logs."),
    71: ("Multi-region failover configuration",
        "Multi-region failover is configured per workspace. Set a primary and "
        "secondary region in Settings > Reliability > Failover. The platform "
        "replicates state asynchronously with a typical RPO of 60 seconds. "
        "Failover is manually triggered; there is no automatic regional cutover."),
    102: ("Billing alerts and budget thresholds",
        "Set billing alerts under Settings > Billing > Budgets. Each budget "
        "supports up to five notification thresholds at percent values of the "
        "monthly cap. Notifications go to the workspace owner email and any "
        "additional recipients you add explicitly."),
}
for i in range(200):
    page = (i // 5) + 1
    if i in seeded_topics:
        title, content = seeded_topics[i]
    else:
        title = f"Section {page}.{(i % 5) + 1}"
        content = (
            f"Section {page}.{(i % 5) + 1} of the product manual. This passage "
            f"covers configuration option {i} and includes references to related "
            f"sections elsewhere in the manual. Generic product content."
        )
    CORPUS_CHUNKS.append({
        "id": f"chunk-{i:03d}",
        "title": title,
        "content": content,
        "source_page": page,
    })
print(f"Seeded {len(CORPUS_CHUNKS)} corpus chunks.")

# AzureOpenAI client with DefaultAzureCredential.
token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

print("Embedding 200 docs into 1536-dim vectors in three batches...")
# YOUR CODE: Embed every chunk's content in one batched call via
# embed_response = aoai_client.embeddings.create(
#     model=EMBED_DEPLOYMENT_NAME,
#     input=[c["content"] for c in CORPUS_CHUNKS],
# )
# The response object stays named embed_response so the upload loop below works.

# Attach each embedding to its chunk.
chunks_with_vectors = [
    {**chunk, "content_vector": datum.embedding}
    for chunk, datum in zip(CORPUS_CHUNKS, embed_response.data)
]

# Upload to the index in batches of 50.
search_data_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AZURE_CREDENTIAL,
)
total_uploaded = 0
for start in range(0, len(chunks_with_vectors), 50):
    batch = chunks_with_vectors[start:start + 50]
    upload_result = search_data_client.upload_documents(documents=batch)
    total_uploaded += sum(1 for r in upload_result if r.succeeded)
print(f"Uploaded {total_uploaded} of {len(chunks_with_vectors)} documents.")

print("Asking Search Basic to wake up and commit the index, gives it about a minute...")
for _ in range(20):
    count_now = search_data_client.get_document_count()
    if count_now >= len(chunks_with_vectors):
        break
    time.sleep(3)
print(f"Index now reports {search_data_client.get_document_count()} documents.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Search Basic service exists in eastus with the lab tag,
# provisioningState=Succeeded; the index has the required fields, a 1536-dim
# vector with an HNSW profile, a semantic configuration; and the index holds
# 200 documents.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        sm = SearchManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
        try:
            svc = sm.services.get(RESOURCE_GROUP, SEARCH_SERVICE_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search service {SEARCH_SERVICE_NAME} does not exist in "
                    f"{RESOURCE_GROUP}. Did the begin_create_or_update poller return?"
                ),
            )

        sku_name = (svc.sku.name or "").lower() if svc.sku else ""
        if sku_name != "basic":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search SKU is {sku_name!r}, expected 'basic'. Free has no real "
                    f"vector profile; Standard or higher is cost overkill for a 200-doc lab."
                ),
            )
        if (svc.location or "").lower() != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=f"Search location is {svc.location!r}, expected {REGION!r}.",
            )
        tags = svc.tags or {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search service is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {tags}."
                ),
            )
        if (svc.provisioning_state or "").lower() != "succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Search provisioningState is {svc.provisioning_state!r}, "
                    f"expected 'succeeded'."
                ),
            )

        sic = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=AZURE_CREDENTIAL)
        try:
            idx = sic.get_index(INDEX_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Index {INDEX_NAME} does not exist on {SEARCH_SERVICE_NAME}. "
                    f"Did create_or_update_index run?"
                ),
            )

        by_name = {f.name: f for f in idx.fields}
        for required in ("id", "title", "content", "content_vector"):
            if required not in by_name:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Index is missing field {required!r}. Found: "
                        f"{sorted(by_name.keys())}"
                    ),
                )

        vec = by_name["content_vector"]
        if getattr(vec, "vector_search_dimensions", None) != EMBED_DIMENSIONS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"content_vector dimensions are "
                    f"{getattr(vec, 'vector_search_dimensions', None)!r}, "
                    f"expected {EMBED_DIMENSIONS}. text-embedding-3-small returns 1536-dim "
                    f"vectors."
                ),
            )
        if not getattr(vec, "vector_search_profile_name", None):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "content_vector field has no vector_search_profile_name set."
                ),
            )

        if not idx.vector_search or not idx.vector_search.algorithms:
            return CheckpointResult(
                status="fail",
                error_reason="Index has no vector_search.algorithms. Declare HNSW.",
            )
        algo_names = [getattr(a, "name", "") for a in idx.vector_search.algorithms]
        if "hnsw-config" not in algo_names:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No HnswAlgorithmConfiguration named 'hnsw-config' in the index. "
                    f"Got: {algo_names}."
                ),
            )
        profile_names = [getattr(p, "name", "") for p in (idx.vector_search.profiles or [])]
        if "vector-profile" not in profile_names:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No VectorSearchProfile named 'vector-profile' in the index. "
                    f"Got: {profile_names}."
                ),
            )

        if not idx.semantic_search or not idx.semantic_search.configurations:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Index has no semantic_search.configurations. Add a "
                    "SemanticConfiguration named 'default'."
                ),
            )
        cfg_by_name = {c.name: c for c in idx.semantic_search.configurations}
        sc = cfg_by_name.get("default")
        if sc is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No SemanticConfiguration named 'default'. Got: "
                    f"{sorted(cfg_by_name.keys())}."
                ),
            )
        pf = getattr(sc, "prioritized_fields", None)
        title_field = getattr(pf, "title_field", None) if pf else None
        content_fields = getattr(pf, "content_fields", None) or [] if pf else []
        if not title_field or getattr(title_field, "field_name", None) != "title":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Semantic configuration 'default' does not name 'title' as the "
                    "title field."
                ),
            )
        if not any(getattr(f, "field_name", None) == "content" for f in content_fields):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Semantic configuration 'default' does not include 'content' as "
                    "a content field."
                ),
            )

        sdc = SearchClient(
            endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=AZURE_CREDENTIAL,
        )
        try:
            count = sdc.get_document_count()
        except HttpResponseError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"get_document_count raised HttpResponseError: {e.message}",
            )
        if count != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Index document count is {count}, expected 200. Did the upload "
                    f"complete and the index commit settle?"
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The validator surfaces the exact failing piece: SKU, location, tag, provisioning state, missing field, profile name, semantic configuration, or document count. The 200-doc count is the most common miss; if it reports 0 the index commit has not settled, or the embedding call did not run. Look at the cell output for the line that reports `Embedding 200 docs into 1536-dim vectors`.

</details>

<details><summary>Hint 2 (direction)</summary>

One call. `aoai_client.embeddings.create(model=EMBED_DEPLOYMENT_NAME, input=[c["content"] for c in CORPUS_CHUNKS])` returns an object whose `.data` is a parallel list of embedding records. Each `.data[i].embedding` is the 1536-dim vector for `CORPUS_CHUNKS[i]`. The variable name `embed_response` is what the upload loop below already references.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 1:

```python
embed_response = aoai_client.embeddings.create(
    model=EMBED_DEPLOYMENT_NAME,
    input=[c["content"] for c in CORPUS_CHUNKS],
)
```

</details>


**Wallet check.** Search Basic is now ticking at $0.10/hour. Embedding 200 chunks at roughly 80 tokens each is about 16,000 tokens, which at $0.02 per 1M tokens is around $0.0003. The Foundry hub, project, and Standard deployment carry no hourly fee at zero traffic. Total damage so far: a couple cents on Search, fractions of a cent on embeddings. Your morning coffee is still ahead but the lead is shrinking.


## Task 2: Mode A vector-only. Run the same 5 queries through `vector_queries=[...]` with `search_text=""` and capture top-3

This is the `code_observe` hybrid. The CODE phase issues the queries. The OBSERVE phase prints the captured scores so you can see what vector-only retrieval looks like before adding BM25 or the ranker.

Build it in this order:

1. Helper function `embed_query(text)` that hits the same `text-embedding-3-small` deployment and returns the 1536-dim vector. Use this helper for every mode so the comparison is fair.
2. Loop the five queries in `TEST_QUERIES`. For each, embed the text, then call `search_data_client.search(search_text="", vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=3, fields="content_vector")], top=3)`.
3. Materialise each iterator and store the top-3 list (with `@search.score`, `id`, `title`, `content`) in `MODE_A_RESULTS[query_id]`.

**Why `search_text=""` not omitted.** Passing the empty string is how the SDK signals "pure vector, no BM25 contribution." Omitting `search_text` entirely is equivalent on most SDK versions, but the empty-string form is the documented and unambiguous shape.

**Why capture the same fields across all three modes.** The comparison table at the end is per-row identical across modes; the scoring field is what changes between Mode A, B, and C.


In [ ]:
# NBVAL_SKIP
# Task 2 (CODE phase): Mode A vector-only. Run the 5 queries through
# vector_queries=[...] with search_text="" and capture top-3 per query.

def embed_query(text: str) -> list[float]:
    """Single shared embedding helper. All three modes use this."""
    response = aoai_client.embeddings.create(
        model=EMBED_DEPLOYMENT_NAME, input=[text],
    )
    return response.data[0].embedding


MODE_A_RESULTS = {}
for q in TEST_QUERIES:
    qv = embed_query(q["text"])
    print(f"Running query {q['id']} ({q['text']!r}) through vector-only mode...")
    # YOUR CODE: Run the vector-only search via
    # results_iter = search_data_client.search(
    #     search_text="",
    #     vector_queries=[VectorizedQuery(
    #         vector=qv,
    #         k_nearest_neighbors=3,
    #         fields="content_vector",
    #     )],
    #     top=3,
    # )
    # Materialise as a list and store under the query id.
    top3 = list(results_iter)
    MODE_A_RESULTS[q["id"]] = [
        {
            "id": r.get("id"),
            "title": r.get("title"),
            "content": r.get("content"),
            "score": r.get("@search.score"),
        }
        for r in top3
    ]

print(f"Captured Mode A results for {len(MODE_A_RESULTS)} queries.")


In [ ]:
# NBVAL_SKIP
# Task 2 (OBSERVE phase): pretty-print Mode A captures so you can read the
# numbers before stacking the comparison table.

print(f"{'query_id':<8} | {'top_1_doc_id':<12} | {'top_1_score':>11} | {'top_3_mean':>11}")
print("-" * 56)
for q in TEST_QUERIES:
    rows = MODE_A_RESULTS.get(q["id"], [])
    if not rows:
        print(f"{q['id']:<8} | {'(no results)':<12} | {'-':>11} | {'-':>11}")
        continue
    top1 = rows[0]
    scores = [r["score"] for r in rows if r["score"] is not None]
    mean_score = statistics.mean(scores) if scores else 0.0
    print(
        f"{q['id']:<8} | {str(top1['id'] or '-'):<12} | "
        f"{top1['score']:>11.4f} | {mean_score:>11.4f}"
    )


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: MODE_A_RESULTS has all 5 query ids; each value is a non-empty
# list; every item has a numeric @search.score (captured as 'score'). Section
# 21 strict: NO comparison logic, this is infrastructure validation.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(MODE_A_RESULTS, dict):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "MODE_A_RESULTS is not a dict. Initialise it as {} at the top of "
                    "the cell and assign per-query lists to MODE_A_RESULTS[q['id']]."
                ),
            )
        for q in TEST_QUERIES:
            qid = q["id"]
            if qid not in MODE_A_RESULTS:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_A_RESULTS is missing query {qid!r}. Did the loop iterate "
                        f"every entry in TEST_QUERIES?"
                    ),
                )
            rows = MODE_A_RESULTS[qid]
            if not isinstance(rows, list) or len(rows) < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_A_RESULTS[{qid!r}] is empty. The vector-only query for "
                        f"{q['text']!r} returned no documents. Confirm the search call "
                        f"was issued with vector_queries=[...] and the embedding helper "
                        f"returned a 1536-dim vector."
                    ),
                )
            for i, r in enumerate(rows, start=1):
                if not isinstance(r, dict):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_A_RESULTS[{qid!r}][{i-1}] is not a dict. Capture each "
                            f"result as {{'id': ..., 'score': r['@search.score'], ...}}."
                        ),
                    )
                score = r.get("score")
                if not isinstance(score, (int, float)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_A_RESULTS[{qid!r}][{i-1}] has no numeric 'score' "
                            f"field. Capture r['@search.score'] from each result."
                        ),
                    )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

Vector-only means `search_text=""` AND a `vector_queries=[...]` list with one `VectorizedQuery`. If results come back empty for every query, the embedding helper probably returned the wrong shape; confirm `len(qv) == 1536`. If results come back only for some queries, the corpus does not cover that topic; vector-only is the most sensitive mode to topic coverage gaps.

</details>

<details><summary>Hint 2 (direction)</summary>

One call. `search_data_client.search(search_text="", vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=3, fields="content_vector")], top=3)` returns an iterator. Materialise with `list(...)` before assigning to `MODE_A_RESULTS[q["id"]]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 2:

```python
results_iter = search_data_client.search(
    search_text="",
    vector_queries=[VectorizedQuery(
        vector=qv,
        k_nearest_neighbors=3,
        fields="content_vector",
    )],
    top=3,
)
```

</details>


**Wallet check.** Five query embeddings at roughly 8 tokens each is about 40 tokens total, well under a thousandth of a cent. The search calls themselves are part of the Basic-tier hourly fee, so they cost nothing on top. Search Basic continues ticking at $0.10/hour. Coffee math still holds.


## Task 3: Mode B hybrid. Pass both `search_text=<text>` and `vector_queries=[...]` and capture top-3

Mode B is the same shape as Mode A with one change: `search_text` carries the literal query string. When both `search_text` and `vector_queries` are present the SDK runs BM25 keyword search and HNSW vector search in parallel, then fuses the two ranked lists via Reciprocal Rank Fusion (RRF) into a single combined ranking. You do not call RRF yourself; the SDK does it implicitly.

Build it in this order:

1. Reuse `embed_query` (same helper, same model, same shape).
2. Loop the five queries. For each, embed the text, then call `search_data_client.search(search_text=q["text"], vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=3, fields="content_vector")], top=3)`.
3. Materialise the top-3 and store in `MODE_B_RESULTS[query_id]`.

**Why implicit RRF.** RRF combines two ranked lists by summing `1/(k + rank)` for each document, where `k` is a small constant (60 by Microsoft's implementation). You do not need to compute this yourself; passing both `search_text` and `vector_queries` triggers it. Students who try to manually merge two separate result lists duplicate work and end up with a worse ranking.

**Why the captured shape is identical to Mode A.** The comparison table is per-row identical across modes. Mode B has the same `score` field semantics as Mode A from the captured-rows perspective; only the underlying retrieval algorithm differs.


In [ ]:
# NBVAL_SKIP
# Task 3 (CODE phase): Mode B hybrid. Pass both search_text=<text> AND
# vector_queries=[...]. The SDK does RRF fusion implicitly.

MODE_B_RESULTS = {}
for q in TEST_QUERIES:
    qv = embed_query(q["text"])
    print(f"Running query {q['id']} ({q['text']!r}) through hybrid mode...")
    # YOUR CODE: Run the hybrid search via
    # results_iter = search_data_client.search(
    #     search_text=q["text"],
    #     vector_queries=[VectorizedQuery(
    #         vector=qv,
    #         k_nearest_neighbors=3,
    #         fields="content_vector",
    #     )],
    #     top=3,
    # )
    # Materialise as a list and store under the query id.
    top3 = list(results_iter)
    MODE_B_RESULTS[q["id"]] = [
        {
            "id": r.get("id"),
            "title": r.get("title"),
            "content": r.get("content"),
            "score": r.get("@search.score"),
        }
        for r in top3
    ]

print(f"Captured Mode B results for {len(MODE_B_RESULTS)} queries.")


In [ ]:
# NBVAL_SKIP
# Task 3 (OBSERVE phase): pretty-print Mode B captures alongside the Mode A
# numbers for the same query ids. You will assemble the full 15-row table
# after Mode C.

print(f"{'query_id':<8} | {'top_1_doc_id':<12} | {'top_1_score':>11} | {'top_3_mean':>11}")
print("-" * 56)
for q in TEST_QUERIES:
    rows = MODE_B_RESULTS.get(q["id"], [])
    if not rows:
        print(f"{q['id']:<8} | {'(no results)':<12} | {'-':>11} | {'-':>11}")
        continue
    top1 = rows[0]
    scores = [r["score"] for r in rows if r["score"] is not None]
    mean_score = statistics.mean(scores) if scores else 0.0
    print(
        f"{q['id']:<8} | {str(top1['id'] or '-'):<12} | "
        f"{top1['score']:>11.4f} | {mean_score:>11.4f}"
    )


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: MODE_B_RESULTS has all 5 query ids; each value is a non-empty
# list; every item has a numeric score. Section 21 strict: NO comparison logic,
# the comparison lives in the reflection MCQ.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(MODE_B_RESULTS, dict):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "MODE_B_RESULTS is not a dict. Initialise it as {} at the top of "
                    "the cell and assign per-query lists to MODE_B_RESULTS[q['id']]."
                ),
            )
        for q in TEST_QUERIES:
            qid = q["id"]
            if qid not in MODE_B_RESULTS:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_B_RESULTS is missing query {qid!r}. Did the loop iterate "
                        f"every entry in TEST_QUERIES?"
                    ),
                )
            rows = MODE_B_RESULTS[qid]
            if not isinstance(rows, list) or len(rows) < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_B_RESULTS[{qid!r}] is empty. The hybrid query for "
                        f"{q['text']!r} returned no documents. Confirm BOTH "
                        f"search_text=q['text'] AND vector_queries=[...] are passed on "
                        f"the same search call."
                    ),
                )
            for i, r in enumerate(rows, start=1):
                if not isinstance(r, dict):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_B_RESULTS[{qid!r}][{i-1}] is not a dict."
                        ),
                    )
                score = r.get("score")
                if not isinstance(score, (int, float)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_B_RESULTS[{qid!r}][{i-1}] has no numeric 'score' "
                            f"field. Capture r['@search.score'] from each result."
                        ),
                    )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

Hybrid means BOTH `search_text=q["text"]` AND `vector_queries=[...]` on the same call. The SDK fuses the two ranked lists with Reciprocal Rank Fusion automatically; you do not need to compute the fusion yourself. If a query returns zero results, you probably passed `search_text=""` (which collapses to Mode A) or omitted `vector_queries=[...]` (which collapses to pure BM25).

</details>

<details><summary>Hint 2 (direction)</summary>

One call. `search_data_client.search(search_text=q["text"], vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=3, fields="content_vector")], top=3)`. The `top=3` parameter applies to the post-fusion ranked list, not to each component.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 3:

```python
results_iter = search_data_client.search(
    search_text=q["text"],
    vector_queries=[VectorizedQuery(
        vector=qv,
        k_nearest_neighbors=3,
        fields="content_vector",
    )],
    top=3,
)
```

</details>


**Wallet check.** Another five query embeddings at roughly 8 tokens each is another fraction of a thousandth of a cent. The helper recomputes the query embedding for each mode because the helper is the consistent path; the embedding model is the same, the cost is identical. Search Basic continues ticking. Coffee math holds.


## Task 4: Mode C hybrid plus semantic ranker. Build the 15-row comparison table

Mode C is Mode B plus two extra keyword arguments: `query_type="semantic"` and `semantic_configuration_name="default"`. The semantic ranker reorders the top-50 candidates returned by retrieval using a deeper LLM-style relevance score. Every Mode C result carries a `@search.reranker_score`. That field is the only honest signal the ranker actually ran; if it is missing, the call silently fell through to non-semantic ranking.

Build it in this order:

1. Reuse `embed_query` (same helper, same model).
2. Loop the five queries. For each, embed the text, then call `search_data_client.search(search_text=q["text"], vector_queries=[VectorizedQuery(...)], query_type="semantic", semantic_configuration_name="default", top=3)`.
3. Materialise the top-3 and store in `MODE_C_RESULTS[query_id]`. Capture BOTH `@search.score` and `@search.reranker_score` on each result.
4. Build `COMPARISON_TABLE`: a list of 15 dicts (5 queries x 3 modes) with keys `query_id`, `mode`, `top_1_score`, `top_3_mean_score`, `top_1_doc_id`. For Mode C, use `reranker_score` as the score field; for Modes A and B, use `score`.
5. Compute and print per-mode mean top-1: Mode A vector mean, Mode B hybrid mean, Mode C reranker mean. This is the `comparisonMetric` string the Option D Colab card surfaces on pass.

**Why `reranker_score` is on a 0-4 scale.** Microsoft's semantic ranker scores reranked documents on a 0-to-4 scale (4 is highly relevant, 0 is not relevant). The plain `@search.score` for vector and hybrid lives in roughly the 0-to-1 range for vector and a wider RRF-fused range for hybrid. The three score ranges are NOT directly comparable across modes; the comparison table reports them side by side anyway because the SHAPE of the numbers is what the production-default argument is about, not absolute values.

**Why no comparison logic in the validator.** Per blueprint Section 21, Compare-it checkpoints validate infrastructure (the modes ran, the captures are shaped correctly, the table has 15 rows). The comparative reasoning ("which mode wins") is reflection-cell territory, surfaced through the exam-style MCQ.


In [ ]:
# NBVAL_SKIP
# Task 4 (CODE phase): Mode C hybrid + semantic ranker. Capture both
# @search.score AND @search.reranker_score on every result.

MODE_C_RESULTS = {}
for q in TEST_QUERIES:
    qv = embed_query(q["text"])
    print(f"Running query {q['id']} of {len(TEST_QUERIES)} through hybrid + semantic mode...")
    # YOUR CODE: Run the hybrid + semantic-ranked search via
    # results_iter = search_data_client.search(
    #     search_text=q["text"],
    #     vector_queries=[VectorizedQuery(
    #         vector=qv,
    #         k_nearest_neighbors=3,
    #         fields="content_vector",
    #     )],
    #     query_type="semantic",
    #     semantic_configuration_name="default",
    #     top=3,
    # )
    # Materialise as a list and store under the query id.
    top3 = list(results_iter)
    MODE_C_RESULTS[q["id"]] = [
        {
            "id": r.get("id"),
            "title": r.get("title"),
            "content": r.get("content"),
            "score": r.get("@search.score"),
            "reranker_score": r.get("@search.reranker_score"),
        }
        for r in top3
    ]

print(f"Captured Mode C results for {len(MODE_C_RESULTS)} queries.")
print("Building the 15-row comparison table from the captured scores...")

COMPARISON_TABLE = []


def _mean(values):
    nums = [v for v in values if isinstance(v, (int, float))]
    return statistics.mean(nums) if nums else 0.0


for q in TEST_QUERIES:
    qid = q["id"]
    a_rows = MODE_A_RESULTS.get(qid, [])
    b_rows = MODE_B_RESULTS.get(qid, [])
    c_rows = MODE_C_RESULTS.get(qid, [])
    if a_rows:
        COMPARISON_TABLE.append({
            "query_id": qid,
            "mode": "vector",
            "top_1_score": a_rows[0].get("score"),
            "top_3_mean_score": _mean([r.get("score") for r in a_rows]),
            "top_1_doc_id": a_rows[0].get("id"),
        })
    if b_rows:
        COMPARISON_TABLE.append({
            "query_id": qid,
            "mode": "hybrid",
            "top_1_score": b_rows[0].get("score"),
            "top_3_mean_score": _mean([r.get("score") for r in b_rows]),
            "top_1_doc_id": b_rows[0].get("id"),
        })
    if c_rows:
        COMPARISON_TABLE.append({
            "query_id": qid,
            "mode": "hybrid+semantic",
            # Mode C uses reranker_score as the comparison-side score field.
            "top_1_score": c_rows[0].get("reranker_score"),
            "top_3_mean_score": _mean([r.get("reranker_score") for r in c_rows]),
            "top_1_doc_id": c_rows[0].get("id"),
        })

print(f"COMPARISON_TABLE assembled with {len(COMPARISON_TABLE)} rows.")


In [ ]:
# NBVAL_SKIP
# Task 4 (OBSERVE phase): pretty-print the 15-row COMPARISON_TABLE and compute
# the per-mode mean top-1 summary. The summary string is what the Option D
# Colab card renders on PASS as comparisonMetric.

print(
    f"{'query_id':<8} | {'mode':<16} | {'top_1_score':>11} | "
    f"{'top_3_mean':>11} | {'top_1_doc_id':<12}"
)
print("-" * 72)
for row in COMPARISON_TABLE:
    ts = row.get("top_1_score")
    tm = row.get("top_3_mean_score")
    ts_str = f"{ts:.4f}" if isinstance(ts, (int, float)) else "-"
    tm_str = f"{tm:.4f}" if isinstance(tm, (int, float)) else "-"
    print(
        f"{row['query_id']:<8} | {row['mode']:<16} | {ts_str:>11} | "
        f"{tm_str:>11} | {str(row.get('top_1_doc_id') or '-'):<12}"
    )

print()
print("Comparison summary (per-mode mean top-1 across all 5 queries):")
vec_mean = _mean([
    r["top_1_score"] for r in COMPARISON_TABLE if r["mode"] == "vector"
])
hyb_mean = _mean([
    r["top_1_score"] for r in COMPARISON_TABLE if r["mode"] == "hybrid"
])
sem_mean = _mean([
    r["top_1_score"] for r in COMPARISON_TABLE if r["mode"] == "hybrid+semantic"
])
print(
    f"Vector mean top-1: {vec_mean:.4f}, "
    f"Hybrid mean top-1: {hyb_mean:.4f}, "
    f"Hybrid+Semantic mean reranker: {sem_mean:.4f}."
)
print()
print("Score ranges are not directly comparable across modes. Vector scores live")
print("in roughly the 0-to-1 range. Hybrid scores carry RRF fusion magnitude.")
print("Reranker scores live on a documented 0-to-4 scale. Read the SHAPE of the")
print("numbers, not the raw magnitudes.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: MODE_C_RESULTS has all 5 query ids; each value is a non-empty
# list; every item has BOTH a numeric 'score' AND a numeric 'reranker_score'.
# COMPARISON_TABLE has exactly 15 rows with all required fields populated.
# Section 21 strict: NO comparison logic; only structural assertions.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(MODE_C_RESULTS, dict):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "MODE_C_RESULTS is not a dict. Initialise it as {} at the top of "
                    "the cell."
                ),
            )
        for q in TEST_QUERIES:
            qid = q["id"]
            if qid not in MODE_C_RESULTS:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_C_RESULTS is missing query {qid!r}."
                    ),
                )
            rows = MODE_C_RESULTS[qid]
            if not isinstance(rows, list) or len(rows) < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"MODE_C_RESULTS[{qid!r}] is empty. The hybrid + semantic query "
                        f"for {q['text']!r} returned no documents."
                    ),
                )
            for i, r in enumerate(rows, start=1):
                if not isinstance(r, dict):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_C_RESULTS[{qid!r}][{i-1}] is not a dict."
                        ),
                    )
                if not isinstance(r.get("score"), (int, float)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"MODE_C_RESULTS[{qid!r}][{i-1}] has no numeric 'score' "
                            f"field. Capture r['@search.score'] from each result."
                        ),
                    )
                if not isinstance(r.get("reranker_score"), (int, float)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            "Mode C returned results without @search.reranker_score. "
                            "Add `query_type='semantic'` AND "
                            "`semantic_configuration_name='default'` to the search call."
                        ),
                    )

        if not isinstance(COMPARISON_TABLE, list):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "COMPARISON_TABLE is not a list. Build it as a list of 15 dicts."
                ),
            )
        if len(COMPARISON_TABLE) != 15:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"COMPARISON_TABLE has {len(COMPARISON_TABLE)} rows, expected 15 "
                    f"(5 queries x 3 modes)."
                ),
            )
        required_keys = {"query_id", "mode", "top_1_score", "top_3_mean_score", "top_1_doc_id"}
        valid_modes = {"vector", "hybrid", "hybrid+semantic"}
        for i, row in enumerate(COMPARISON_TABLE):
            if not isinstance(row, dict):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"COMPARISON_TABLE[{i}] is not a dict.",
                )
            missing = required_keys - set(row.keys())
            if missing:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"COMPARISON_TABLE[{i}] is missing keys {sorted(missing)}. "
                        f"Each row needs query_id, mode, top_1_score, top_3_mean_score, "
                        f"top_1_doc_id."
                    ),
                )
            if row["mode"] not in valid_modes:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"COMPARISON_TABLE[{i}] mode is {row['mode']!r}, expected one "
                        f"of {sorted(valid_modes)}."
                    ),
                )
            if not isinstance(row.get("top_1_score"), (int, float)):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"COMPARISON_TABLE[{i}] top_1_score is not numeric."
                    ),
                )
            if not isinstance(row.get("top_3_mean_score"), (int, float)):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"COMPARISON_TABLE[{i}] top_3_mean_score is not numeric."
                    ),
                )
            if not row.get("top_1_doc_id"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"COMPARISON_TABLE[{i}] top_1_doc_id is missing or empty."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The reranker_score is the only honest signal the ranker actually ran. If a Mode C result has no `@search.reranker_score`, the query silently fell through to non-semantic ranking, usually because one of the two required kwargs was missing. Look at the failing validator message; it will tell you which kwarg is the culprit.

</details>

<details><summary>Hint 2 (direction)</summary>

Two kwargs together: `query_type="semantic"` AND `semantic_configuration_name="default"`. Adding only one returns a 400 (or silently drops the ranker, depending on SDK version). Both are required and both must reference an existing configuration on the index. The `default` name matches the SemanticConfiguration declared in Task 1.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 4:

```python
results_iter = search_data_client.search(
    search_text=q["text"],
    vector_queries=[VectorizedQuery(
        vector=qv,
        k_nearest_neighbors=3,
        fields="content_vector",
    )],
    query_type="semantic",
    semantic_configuration_name="default",
    top=3,
)
```

</details>


**Wallet check.** Another five query embeddings, another fraction of a thousandth of a cent. The total session cost is dominated by the Search Basic hourly bill. If you have been about 30 to 45 minutes into the session, you are roughly at four to seven cents on Search. Run cleanup as soon as Checkpoint 4 passes.


## Cleanup

Time to tear it all down. **The Search service is the cost-critical resource in this lab.** The cell below runs the manifest in reverse-creation order (index, then search service, then deployment, then project, hub, resource group), then verifies via Resource Graph that nothing tagged with this lab's slug is left. Re-running is safe. If cleanup fails on any resource, `sys.exit(1)` fires so the companion panel marks the session dirty and you have a chance to fix it before walking away.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. The Search service is the only critical resource; deleting it stops
# the $0.10/hour meter.

import sys

# Resolve AOAI_ACCOUNT_NAME into the deployment CLI strings now that it has
# been discovered.
for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "ai_search_service")
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_failed = sum(
    1 for r in CLEANUP_MANIFEST if r.type == "ai_search_service" and r.id in failed_ids
)
standard_failed = len(failed_ids) - critical_failed
critical_destroyed = critical_total - critical_failed
standard_destroyed = standard_total - standard_failed
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")
if critical_failed > 0:
    print()
    print("CRITICAL: an Azure AI Search Basic service is still alive.")
    print(f"That meter is ticking at ${SEARCH_BASIC_HOURLY_USD:.2f}/hour ($73.73/month).")
    print(
        f"Run manually: az search service delete --resource-group {RESOURCE_GROUP} "
        f"--name {SEARCH_SERVICE_NAME} --yes"
    )

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $0.20 to $0.30 depending on session length.** Search Basic at $0.10/hour dominates; `text-embedding-3-small` adds fractions of a cent. The Foundry hub, project, and Standard deployment carry no hourly fee at zero traffic. Two Search labs in this track. Both burn ten cents an hour if you forget to clean up. The point of running the three modes side by side is the comparison table you just captured. The relevance numbers will not be identical from session to session because the corpus is small and the queries are open-ended, but the shape of the comparison should be reproducible. That shape is what the production-default argument is about. Check Azure Cost Management in 24 hours to confirm the exact number.


## Reflection

These are not graded. They are for you.

1. The comparison table is captured. For your team's hypothetical production workload (mostly natural-language questions, ~20% exact-SKU lookups), which mode would you choose as the default and why? What signal in your captured table supports the choice? What signal would push you to change your mind in three months?

2. The semantic ranker added a `reranker_score` to the top-3 results in Mode C. Walk through what the ranker actually does internally (it reorders the top-50 candidates returned by retrieval based on a deeper LLM-style relevance scoring) and what failure mode it does NOT catch (for example, if retrieval missed the right doc entirely, no reordering brings it back).

## Exam-Style Practice

**Question 1 (Domain 5, retrieval mode for SKU-dominated workload; constraint-driven comparison per blueprint Section 21):**

A team's production search workload is 70% exact-product-code lookups (e.g., "SKU-A123-X") and 30% natural-language questions. The team must pick ONE retrieval default. The 70% case must work; the 30% case should work as well as possible. Which mode fits?

A. Vector-only with `text-embedding-3-small`; lets semantic embedding handle both cases uniformly.

B. Hybrid (vector + BM25) without the semantic ranker; BM25 gives exact-code lookups while vector covers natural-language semantics.

C. Hybrid + semantic ranker; BM25 + vector cover retrieval, the ranker reorders for relevance.

D. Pure keyword (BM25) search; exact-code lookups are the dominant case.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: vector-only is the weakest mode for exact-code lookups. Embeddings of "SKU-A123-X" are not strongly distinct from "SKU-A124-X" in the vector space; BM25 token-matches the exact code reliably.
- B is partially right but misses the relevance reordering on the 30% case. Hybrid alone is a good baseline but the semantic ranker reorders the top-50 candidates by query intent, which improves the 30% natural-language case at modest cost.
- C is correct: the AI-103 documented production-RAG pattern. BM25 handles SKU codes, vector handles semantic, the ranker reorders for the 30% natural-language case. Best of all three on a mixed workload.
- D is wrong: keyword-only abandons the 30% natural-language case. The team explicitly said both cases matter.

</details>

**Question 2 (Domain 5, captured-measurement comparison):**

In a vector-vs-hybrid-vs-semantic comparison, a team observes that for a specific query the top result's document ID is the SAME across all three modes, but the `@search.score` values are different (vector: 0.84, hybrid: 0.92, semantic + reranker: reranker_score 2.7). What is the right takeaway from this single-query observation?

A. The hybrid mode is objectively better because 0.92 > 0.84.

B. The semantic ranker is broken because 2.7 is outside the 0-1 range of the other scores.

C. The score ranges are not directly comparable across modes; the right takeaway is that all three modes surfaced the same top doc for this query, which is a positive signal, but single-query observations do not generalize. The team needs a 5+ query comparison and ideally a reference-answer set to score precision and recall against.

D. The vector score of 0.84 is below 0.9 and therefore the result is unusable.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: `@search.score` values are not normalised across retrieval modes. Hybrid scores include the RRF fusion contribution; vector scores are cosine-similarity-derived; reranker scores have their own 0-4 scale. You cannot directly compare across modes.
- B is wrong: the reranker score is on a 0-4 scale by Microsoft's documented design; 2.7 is a healthy mid-to-high score.
- C is correct: the comparison table is a starting point. Single-query observations should not generalize. Production decisions need a representative query set and ideally graded relevance reference answers. The AI-103 study guide's "evaluate relevance" task statement points here.
- D is wrong: 0.84 vector-similarity is a strong signal in absolute terms; the 0.9 threshold is arbitrary.

</details>
